In [ ]:
# ============================================================================
# CELL 1: SETUP AND DEPENDENCIES
# ============================================================================

!pip -q install sacrebleu sentencepiece

import math
import os
import random
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF, TER
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import sentencepiece as spm

print("PyTorch:", torch.__version__)
print("SacreBLEU:", sacrebleu.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.7 MB/s eta 0:00:00
PyTorch: 2.9.0+cu126
SacreBLEU: 2.5.1
Using device: cuda


In [ ]:
# ============================================================================
# CELL 2: REPRODUCIBILITY AND CONFIGURATION
# ============================================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Language and file config
SRC_LANG = "ur"  # Urdu
TGT_LANG = "en"  # English

SRC_FILE = "Urdu.txt"
TGT_FILE = "English.txt"

# IMPROVED Model/training hyperparameters (optimized for idioms)
MAX_SEQ_LEN = 60           # Increased from 50 for longer idioms
BATCH_SIZE = 32            # Reduced for better generalization
EMBED_DIM = 512            # Increased from 384
N_HEAD = 8
FFN_DIM = 2048             # Increased from 768
NUM_ENCODER_LAYERS = 6     # Increased from 3
NUM_DECODER_LAYERS = 6     # Increased from 3
DROPOUT = 0.1              # Reduced from 0.2

NUM_EPOCHS = 15            # Increased with early stopping (50)
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.1
LEARNING_RATE = 2e-4       # FIXED: Stable learning rate

# SentencePiece vocab sizes
SRC_SP_VOCAB = 8000        # Increased from 6000 for better idiom coverage
TGT_SP_VOCAB = 8000        # Increased from 6000

# Training flags
USE_MIXED_PRECISION = True if torch.cuda.is_available() else False
USE_DATA_AUGMENTATION = True
EARLY_STOPPING_PATIENCE = 5

print("✓ Config ready (Optimized for idioms)")


✓ Config ready (Optimized for idioms)


In [ ]:
# ============================================================================
# CELL 3: DATA LOADING AND PREPROCESSING
# ============================================================================

REPO_RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "Kheem-Dh/Urdu-to-English-Machine-Translation-using-Seq2Seq-Transformers-Variant-Model/main"
)

# Download files if not present
if not os.path.exists(SRC_FILE):
    !wget -q "{REPO_RAW_BASE}/Urdu.txt" -O Urdu.txt
if not os.path.exists(TGT_FILE):
    !wget -q "{REPO_RAW_BASE}/English.txt" -O English.txt

print("Files present?", os.path.exists(SRC_FILE), os.path.exists(TGT_FILE))


def read_parallel_corpus(src_path: str, tgt_path: str) -> Tuple[List[str], List[str]]:
    """Read parallel corpus files"""
    with open(src_path, "r", encoding="utf-8") as f_src, \
         open(tgt_path, "r", encoding="utf-8") as f_tgt:
        src_lines = [l.strip() for l in f_src]
        tgt_lines = [l.strip() for l in f_tgt]

    assert len(src_lines) == len(tgt_lines), \
        "Source and target files must have same number of lines."

    pairs = []
    for s, t in zip(src_lines, tgt_lines):
        if s and t:
            pairs.append((s, t))

    src_clean, tgt_clean = zip(*pairs) if pairs else ([], [])
    return list(src_clean), list(tgt_clean)


src_sentences, tgt_sentences = read_parallel_corpus(SRC_FILE, TGT_FILE)
print(f"Total raw sentence pairs: {len(src_sentences)}")
print("Example pair:")
print("SRC:", src_sentences[0] if src_sentences else "N/A")
print("TGT:", tgt_sentences[0] if tgt_sentences else "N/A")


Files present? True True
Total raw sentence pairs: 100000
Example pair:
SRC: لاس اینجلس نے سیزن شروع کرنے کے لئے سیدھے رات اور اپنے پہلے 14 میں سے 13 کھیل کھوئے ہیں۔
TGT: Los Angeles has lost night straight and 13 of its first 14 games to start the season.


In [ ]:
# ============================================================================
# CELL 4: DATA CLEANING (IMPROVED)
# ============================================================================

print("\n=== Basic length stats BEFORE cleaning ===")
en_lengths = [len(t.split()) for t in tgt_sentences]
print("Total pairs:", len(tgt_sentences))
print("Avg EN length:", np.mean(en_lengths))
print("Median EN length:", np.median(en_lengths))
print("90th percentile EN length:", np.percentile(en_lengths, 90))
print("Max EN length:", max(en_lengths))

MAX_EN_LEN = 60  # Only drop really long outliers
MIN_EN_LEN = 2   # Keep idioms (they can be short)

CLEAN_SRC = []
CLEAN_TGT = []
seen_pairs = set()

for s, t in zip(src_sentences, tgt_sentences):
    s = s.strip()
    t = t.strip()

    if not s or not t:
        continue

    len_t = len(t.split())

    # Remove only extremely long sentences; KEEP short idioms
    if len_t > MAX_EN_LEN or len_t < MIN_EN_LEN:
        continue

    # Remove duplicates
    pair = (s, t)
    if pair in seen_pairs:
        continue
    seen_pairs.add(pair)

    CLEAN_SRC.append(s)
    CLEAN_TGT.append(t)

print("\n=== AFTER cleaning ===")
print("Total clean pairs:", len(CLEAN_SRC))

src_sentences = CLEAN_SRC
tgt_sentences = CLEAN_TGT



=== Basic length stats BEFORE cleaning ===
Total pairs: 100000
Avg EN length: 12.65119
Median EN length: 12.0
90th percentile EN length: 22.0
Max EN length: 27

=== AFTER cleaning ===
Total clean pairs: 90741


In [ ]:
# ============================================================================
# CELL 5: DATA AUGMENTATION (NEW)
# ============================================================================

def augment_data_for_idioms(src_texts: List[str], tgt_texts: List[str],
                             augment_ratio: float = 0.3) -> Tuple[List[str], List[str]]:
    """
    Data augmentation specifically for idioms:
    - Word dropout (helps model focus on key words)
    - No synonym replacement (idioms are fixed expressions)
    """
    augmented_src = []
    augmented_tgt = []

    for s, t in zip(src_texts, tgt_texts):
        # Always keep original
        augmented_src.append(s)
        augmented_tgt.append(t)

        # Apply augmentation probabilistically
        if random.random() < augment_ratio:
            # Word dropout: randomly drop 10% of words
            words = s.split()
            if len(words) > 3:  # Only for longer phrases
                keep_mask = [random.random() > 0.1 for _ in words]
                augmented_s = ' '.join([w for w, keep in zip(words, keep_mask) if keep])

                if augmented_s and len(augmented_s.split()) >= 2:
                    augmented_src.append(augmented_s)
                    augmented_tgt.append(t)

    return augmented_src, augmented_tgt

In [ ]:
# ============================================================================
# CELL 6: TRAIN/VALIDATION/TEST SPLIT
# ============================================================================

def train_val_test_split(src, tgt, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    """Split data into train/val/test sets"""
    assert len(src) == len(tgt)
    n = len(src)
    indices = list(range(n))
    random.shuffle(indices)

    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)

    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]

    def subset(idxs):
        return [src[i] for i in idxs], [tgt[i] for i in idxs]

    src_train, tgt_train = subset(train_idx)
    src_val, tgt_val = subset(val_idx)
    src_test, tgt_test = subset(test_idx)

    print(f"Train: {len(src_train)} | Val: {len(src_val)} | Test: {len(src_test)}")
    return (src_train, tgt_train), (src_val, tgt_val), (src_test, tgt_test)


(train_src, train_tgt), (val_src, val_tgt), (test_src, test_tgt) = \
    train_val_test_split(src_sentences, tgt_sentences)

# Apply augmentation to training data only
if USE_DATA_AUGMENTATION:
    print("\n=== Applying data augmentation ===")
    orig_size = len(train_src)
    train_src, train_tgt = augment_data_for_idioms(train_src, train_tgt, augment_ratio=0.3)
    print(f"Training data: {orig_size} → {len(train_src)} pairs (augmented)")


Train: 72592 | Val: 9074 | Test: 9075

=== Applying data augmentation ===
Training data: 72592 → 93821 pairs (augmented)


In [ ]:
# ============================================================================
# CELL 7: SENTENCEPIECE TOKENIZATION
# ============================================================================

# Prepare corpora for SentencePiece
with open("src_corpus.txt", "w", encoding="utf-8") as f:
    for s in src_sentences:
        f.write(s + "\n")

with open("tgt_corpus.txt", "w", encoding="utf-8") as f:
    for t in tgt_sentences:
        f.write(t + "\n")

# Train source SP model
print("\n=== Training SentencePiece models ===")
spm.SentencePieceTrainer.Train(
    input="src_corpus.txt",
    model_prefix="sp_src",
    vocab_size=SRC_SP_VOCAB,
    model_type="bpe",
    character_coverage=1.0,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
)

# Train target SP model
spm.SentencePieceTrainer.Train(
    input="tgt_corpus.txt",
    model_prefix="sp_tgt",
    vocab_size=TGT_SP_VOCAB,
    model_type="bpe",
    character_coverage=1.0,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
)

sp_src = spm.SentencePieceProcessor()
sp_src.load("sp_src.model")

sp_tgt = spm.SentencePieceProcessor()
sp_tgt.load("sp_tgt.model")

print("Source SentencePiece vocab size:", sp_src.vocab_size())
print("Target SentencePiece vocab size:", sp_tgt.vocab_size())


=== Training SentencePiece models ===
Source SentencePiece vocab size: 8000
Target SentencePiece vocab size: 8000


In [ ]:
# ============================================================================
# CELL 8: VOCABULARY SETUP
# ============================================================================

SPECIAL_TOKENS = {
    "<pad>": 0,
    "<sos>": 1,
    "<eos>": 2,
    "<unk>": 3,
}

PAD_IDX = SPECIAL_TOKENS["<pad>"]
SOS_IDX = SPECIAL_TOKENS["<sos>"]
EOS_IDX = SPECIAL_TOKENS["<eos>"]
UNK_IDX = SPECIAL_TOKENS["<unk>"]

SRC_OFFSET = len(SPECIAL_TOKENS)
TGT_OFFSET = len(SPECIAL_TOKENS)

SRC_VOCAB_SIZE = SRC_OFFSET + sp_src.vocab_size()
TGT_VOCAB_SIZE = TGT_OFFSET + sp_tgt.vocab_size()

print("\nFinal source vocab size:", SRC_VOCAB_SIZE)
print("Final target vocab size:", TGT_VOCAB_SIZE)



Final source vocab size: 8004
Final target vocab size: 8004


In [ ]:
# ============================================================================
# CELL 9: ENCODING AND DECODING FUNCTIONS
# ============================================================================

def encode_src(text: str, max_len: int) -> List[int]:
    """Encode source text to token IDs"""
    sp_ids = sp_src.EncodeAsIds(text)
    sp_ids = sp_ids[: max_len - 2]
    ids = [SOS_IDX] + [i + SRC_OFFSET for i in sp_ids] + [EOS_IDX]
    return ids


def encode_tgt(text: str, max_len: int) -> List[int]:
    """Encode target text to token IDs"""
    sp_ids = sp_tgt.EncodeAsIds(text)
    sp_ids = sp_ids[: max_len - 2]
    ids = [SOS_IDX] + [i + TGT_OFFSET for i in sp_ids] + [EOS_IDX]
    return ids


def decode_tgt_ids(ids: List[int]) -> str:
    """Decode token IDs back to text"""
    pieces = []
    for idx in ids:
        if idx in (PAD_IDX, SOS_IDX, EOS_IDX):
            continue
        if idx < TGT_OFFSET:
            continue
        sp_id = idx - TGT_OFFSET
        if sp_id < sp_tgt.vocab_size():
            pieces.append(sp_tgt.IdToPiece(sp_id))
    text = sp_tgt.DecodePieces(pieces)
    return text


# Quick sanity check
print("\n=== Encoding/Decoding Test ===")
if train_src:
    print("Example encoded SRC:", encode_src(train_src[0], MAX_SEQ_LEN))
    print("Example encoded TGT:", encode_tgt(train_tgt[0], MAX_SEQ_LEN))
    print("Decoded example TGT:", decode_tgt_ids(encode_tgt(train_tgt[0], MAX_SEQ_LEN)))


=== Encoding/Decoding Test ===
Example encoded SRC: [1, 1307, 15, 3633, 22, 69, 45, 5351, 500, 1028, 91, 21, 1558, 535, 1791, 91, 5403, 72, 904, 89, 2]
Example encoded TGT: [1, 475, 56, 237, 947, 393, 3349, 453, 1825, 34, 14, 5038, 38, 14, 1734, 70, 378, 2805, 787, 2532, 7766, 2]
Decoded example TGT: There is also another mysterious line in the middle of the court that other players cannot cross.


In [ ]:
# ============================================================================
# CELL 10: PYTORCH DATASET AND DATALOADER
# ============================================================================

class TranslationDataset(Dataset):
    """Dataset for translation pairs"""
    def __init__(self, src_texts: List[str], tgt_texts: List[str], max_len: int):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.max_len = max_len

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src = self.src_texts[idx]
        tgt = self.tgt_texts[idx]
        src_ids = encode_src(src, self.max_len)
        tgt_ids = encode_tgt(tgt, self.max_len)
        return torch.tensor(src_ids, dtype=torch.long), \
               torch.tensor(tgt_ids, dtype=torch.long)


def collate_fn(batch):
    """Collate function with dynamic padding"""
    src_batch, tgt_batch = zip(*batch)

    src_lens = [len(x) for x in src_batch]
    tgt_lens = [len(x) for x in tgt_batch]
    max_src = max(src_lens)
    max_tgt = max(tgt_lens)

    padded_src = []
    padded_tgt = []

    for src_ids, tgt_ids in zip(src_batch, tgt_batch):
        src_pad = torch.full((max_src,), PAD_IDX, dtype=torch.long)
        tgt_pad = torch.full((max_tgt,), PAD_IDX, dtype=torch.long)
        src_pad[: len(src_ids)] = src_ids
        tgt_pad[: len(tgt_ids)] = tgt_ids
        padded_src.append(src_pad)
        padded_tgt.append(tgt_pad)

    src_tensor = torch.stack(padded_src)
    tgt_tensor = torch.stack(padded_tgt)
    return src_tensor, tgt_tensor


train_dataset = TranslationDataset(train_src, train_tgt, MAX_SEQ_LEN)
val_dataset = TranslationDataset(val_src, val_tgt, MAX_SEQ_LEN)
test_dataset = TranslationDataset(test_src, test_tgt, MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1,
                         shuffle=False, collate_fn=collate_fn)

print("\nBatches - Train:", len(train_loader),
      "Val:", len(val_loader), "Test:", len(test_loader))



Batches - Train: 2932 Val: 284 Test: 9075


In [ ]:
# ============================================================================
# CELL 11: POSITIONAL ENCODING
# ============================================================================

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding"""
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


In [ ]:
# ============================================================================
# CELL 12: IMPROVED TRANSFORMER MODEL
# ============================================================================

class ImprovedSeq2SeqTransformer(nn.Module):
    """
    Improved Transformer with:
    - Pre-norm architecture for stability
    - GELU activation
    - Weight tying
    - Proper initialization
    """
    def __init__(self, num_encoder_layers: int, num_decoder_layers: int,
                 emb_size: int, nhead: int, src_vocab_size: int,
                 tgt_vocab_size: int, dim_feedforward: int = 512,
                 dropout: float = 0.1):
        super().__init__()

        self.emb_size = emb_size

        # Embeddings
        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout)

        # Xavier initialization for embeddings
        nn.init.xavier_uniform_(self.src_tok_emb.weight)
        nn.init.xavier_uniform_(self.tgt_tok_emb.weight)

        # Pre-norm Transformer layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_size,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-norm for better training
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=emb_size,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-norm for better training
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_decoder_layers)

        # Output projection
        self.output_norm = nn.LayerNorm(emb_size)
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

        # Weight tying (reduces parameters)
        self.generator.weight = self.tgt_tok_emb.weight

    def forward(self, src, tgt, src_mask, tgt_mask,
                src_padding_mask, tgt_padding_mask, memory_key_padding_mask):
        # Embeddings with scaling
        src_emb = self.positional_encoding(
            self.src_tok_emb(src) * math.sqrt(self.emb_size)
        )
        tgt_emb = self.positional_encoding(
            self.tgt_tok_emb(tgt) * math.sqrt(self.emb_size)
        )

        # Encode
        memory = self.encoder(src_emb, mask=src_mask,
                            src_key_padding_mask=src_padding_mask)

        # Decode
        output = self.decoder(tgt_emb, memory, tgt_mask=tgt_mask,
                            tgt_key_padding_mask=tgt_padding_mask,
                            memory_key_padding_mask=memory_key_padding_mask)

        # Output projection
        output = self.output_norm(output)
        logits = self.generator(output)
        return logits


def generate_square_subsequent_mask(sz: int) -> torch.Tensor:
    """Generate causal mask for decoder"""
    mask = torch.triu(torch.ones((sz, sz), device=device) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float("-inf")).masked_fill(mask == 1, 0.0)
    return mask


def create_padding_mask(seq: torch.Tensor, pad_idx: int) -> torch.Tensor:
    """Create padding mask"""
    return (seq == pad_idx)


# Initialize model
print("\n=== Initializing Improved Model ===")
model = ImprovedSeq2SeqTransformer(
    NUM_ENCODER_LAYERS,
    NUM_DECODER_LAYERS,
    EMBED_DIM,
    N_HEAD,
    SRC_VOCAB_SIZE,
    TGT_VOCAB_SIZE,
    FFN_DIM,
    DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params/1e6:.2f}M")
print(f"Trainable parameters: {n_trainable/1e6:.2f}M")



=== Initializing Improved Model ===
Model parameters: 52.34M
Trainable parameters: 52.34M


In [ ]:
# ============================================================================
# CELL 13: OPTIMIZER, CRITERION, AND SCHEDULER (FIXED!)
# ============================================================================

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=LABEL_SMOOTHING)

# FIXED: Use stable learning rate with Adam
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE,
                            betas=(0.9, 0.98), eps=1e-9)

# Use ReduceLROnPlateau for stability (instead of warmup)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    #verbose=True,
    min_lr=1e-6
)

print(f"\n✓ Optimizer initialized with lr={LEARNING_RATE}")
print(f"✓ Scheduler: ReduceLROnPlateau (stable)")
print(f"✓ Using mixed precision: {USE_MIXED_PRECISION}")



✓ Optimizer initialized with lr=0.0002
✓ Scheduler: ReduceLROnPlateau (stable)
✓ Using mixed precision: True


In [ ]:
# ============================================================================
# CELL 14: EARLY STOPPING
# ============================================================================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

        return self.should_stop

early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)


In [ ]:
# ============================================================================
# CELL 15: TRAINING AND EVALUATION FUNCTIONS (FIXED!)
# ============================================================================

def train_one_epoch(model, data_loader, optimizer, criterion, use_amp=USE_MIXED_PRECISION):
    """Train for one epoch with NaN detection"""
    model.train()
    total_loss = 0.0
    n_batches = 0
    scaler = GradScaler() if use_amp else None

    for batch_idx, (src, tgt) in enumerate(data_loader):
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        src_seq_len = src.size(1)
        tgt_seq_len = tgt_input.size(1)

        src_mask = torch.zeros((src_seq_len, src_seq_len),
                              device=device).type(torch.bool)
        tgt_mask = generate_square_subsequent_mask(tgt_seq_len)

        src_padding_mask = create_padding_mask(src, PAD_IDX)
        tgt_padding_mask = create_padding_mask(tgt_input, PAD_IDX)

        optimizer.zero_grad()

        if use_amp:
            with autocast():
                logits = model(src, tgt_input, src_mask, tgt_mask,
                             src_padding_mask, tgt_padding_mask, src_padding_mask)
                loss = criterion(logits.reshape(-1, logits.size(-1)),
                               tgt_out.reshape(-1))

            # Check for NaN
            if torch.isnan(loss):
                print(f"⚠️ NaN in batch {batch_idx}! Skipping...")
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(src, tgt_input, src_mask, tgt_mask,
                         src_padding_mask, tgt_padding_mask, src_padding_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)),
                           tgt_out.reshape(-1))

            # Check for NaN
            if torch.isnan(loss):
                print(f"⚠️ NaN in batch {batch_idx}! Skipping...")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


def evaluate_loss(model, data_loader, criterion):
    """Evaluate model on validation/test set"""
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for src, tgt in data_loader:
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_out = tgt[:, 1:]

            src_seq_len = src.size(1)
            tgt_seq_len = tgt_input.size(1)

            src_mask = torch.zeros((src_seq_len, src_seq_len),
                                  device=device).type(torch.bool)
            tgt_mask = generate_square_subsequent_mask(tgt_seq_len)

            src_padding_mask = create_padding_mask(src, PAD_IDX)
            tgt_padding_mask = create_padding_mask(tgt_input, PAD_IDX)

            logits = model(src, tgt_input, src_mask, tgt_mask,
                         src_padding_mask, tgt_padding_mask, src_padding_mask)

            loss = criterion(logits.reshape(-1, logits.size(-1)),
                           tgt_out.reshape(-1))
            total_loss += loss.item()

    return total_loss / len(data_loader)


In [ ]:
# ============================================================================
# CELL 16: TRAINING LOOP (FIXED!)
# ============================================================================

print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80 + "\n")

best_val_loss = float("inf")
train_losses = []
val_losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss = evaluate_loss(model, val_loader, criterion)

    # FIXED: Step scheduler AFTER epoch, not during
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_transformer.pt")
        print(f"✓ Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ★ NEW BEST")
    else:
        print(f"  Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping check
    if early_stopping(val_loss):
        print(f"\n✓ Early stopping triggered at epoch {epoch}")
        break

print("\n" + "="*80)
print("TRAINING FINISHED")
print("="*80)
print(f"Best validation loss: {best_val_loss:.4f}")

# Load best model
model.load_state_dict(torch.load("best_transformer.pt"))
print("✓ Best model loaded")


STARTING TRAINING

✓ Epoch 01 | Train Loss: 5.9157 | Val Loss: 5.1086 | ★ NEW BEST
✓ Epoch 02 | Train Loss: 4.7181 | Val Loss: 4.3122 | ★ NEW BEST
✓ Epoch 03 | Train Loss: 4.0382 | Val Loss: 3.8397 | ★ NEW BEST
✓ Epoch 04 | Train Loss: 3.5833 | Val Loss: 3.5533 | ★ NEW BEST
✓ Epoch 05 | Train Loss: 3.2754 | Val Loss: 3.3850 | ★ NEW BEST
✓ Epoch 06 | Train Loss: 3.0469 | Val Loss: 3.2783 | ★ NEW BEST
✓ Epoch 07 | Train Loss: 2.8674 | Val Loss: 3.2316 | ★ NEW BEST
✓ Epoch 08 | Train Loss: 2.7208 | Val Loss: 3.1829 | ★ NEW BEST
✓ Epoch 09 | Train Loss: 2.5990 | Val Loss: 3.1637 | ★ NEW BEST
✓ Epoch 10 | Train Loss: 2.4936 | Val Loss: 3.1494 | ★ NEW BEST
  Epoch 11 | Train Loss: 2.4050 | Val Loss: 3.1534
  Epoch 12 | Train Loss: 2.3282 | Val Loss: 3.1602
  Epoch 13 | Train Loss: 2.2621 | Val Loss: 3.1531
  Epoch 14 | Train Loss: 2.2031 | Val Loss: 3.1686
✓ Epoch 15 | Train Loss: 1.9945 | Val Loss: 3.1051 | ★ NEW BEST

TRAINING FINISHED
Best validation loss: 3.1051
✓ Best model loaded


In [ ]:
# ============================================================================
# CELL 17: DECODING FUNCTIONS
# ============================================================================

def greedy_decode(model, src, max_len: int = MAX_SEQ_LEN) -> List[int]:
    """Standard greedy decoding"""
    model.eval()
    src = src.to(device)
    src_seq_len = src.size(1)

    src_mask = torch.zeros((src_seq_len, src_seq_len), device=device).type(torch.bool)
    src_padding_mask = create_padding_mask(src, PAD_IDX)

    with torch.no_grad():
        src_emb = model.positional_encoding(model.src_tok_emb(src) * math.sqrt(model.emb_size))
        memory = model.encoder(src_emb, mask=src_mask, src_key_padding_mask=src_padding_mask)

        ys = torch.tensor([[SOS_IDX]], dtype=torch.long, device=device)

        for _ in range(max_len - 1):
            tgt_mask = generate_square_subsequent_mask(ys.size(1))
            tgt_emb = model.positional_encoding(model.tgt_tok_emb(ys) * math.sqrt(model.emb_size))

            out = model.decoder(tgt_emb, memory, tgt_mask=tgt_mask,
                              memory_key_padding_mask=src_padding_mask,
                              tgt_key_padding_mask=create_padding_mask(ys, PAD_IDX))
            out = model.generator(model.output_norm(out[:, -1, :]))
            prob = out.softmax(dim=-1)
            next_token = torch.argmax(prob, dim=-1).item()
            ys = torch.cat([ys, torch.tensor([[next_token]], device=device)], dim=1)

            if next_token == EOS_IDX:
                break

    return ys.squeeze(0).tolist()


def beam_search_decode(model, src: torch.Tensor, beam_size: int = 4,
                      max_len: int = MAX_SEQ_LEN, length_penalty: float = 0.7):
    """Standard beam search"""
    model.eval()
    src = src.to(device)
    src_seq_len = src.size(1)

    src_mask = torch.zeros((src_seq_len, src_seq_len), device=device).type(torch.bool)
    src_padding_mask = create_padding_mask(src, PAD_IDX)

    with torch.no_grad():
        src_emb = model.positional_encoding(model.src_tok_emb(src) * math.sqrt(model.emb_size))
        memory = model.encoder(src_emb, mask=src_mask, src_key_padding_mask=src_padding_mask)

    beams = [(torch.tensor([SOS_IDX], device=device, dtype=torch.long), 0.0)]

    for _ in range(max_len - 1):
        new_beams = []

        for seq, log_prob in beams:
            if seq[-1].item() == EOS_IDX:
                new_beams.append((seq, log_prob))
                continue

            ys = seq.unsqueeze(0)
            tgt_len = ys.size(1)
            tgt_mask = generate_square_subsequent_mask(tgt_len)
            tgt_padding_mask = create_padding_mask(ys, PAD_IDX)

            with torch.no_grad():
                tgt_emb = model.positional_encoding(model.tgt_tok_emb(ys) * math.sqrt(model.emb_size))
                out = model.decoder(tgt_emb, memory, tgt_mask=tgt_mask,
                                  memory_key_padding_mask=src_padding_mask,
                                  tgt_key_padding_mask=tgt_padding_mask)
                out_step = model.generator(model.output_norm(out[:, -1, :]))
                log_probs = F.log_softmax(out_step, dim=-1).squeeze(0)

                log_probs[PAD_IDX] = -1e9
                log_probs[SOS_IDX] = -1e9

                topk_log_probs, topk_ids = torch.topk(log_probs, beam_size)

            for k in range(beam_size):
                next_id = topk_ids[k].item()
                next_log_prob = log_prob + topk_log_probs[k].item()
                new_seq = torch.cat([seq, torch.tensor([next_id], device=device, dtype=torch.long)], dim=0)
                new_beams.append((new_seq, next_log_prob))

        def score_fn(b):
            seq, logp = b
            length = len(seq)
            return logp / (length ** length_penalty)

        new_beams.sort(key=score_fn, reverse=True)
        beams = new_beams[:beam_size]

        if all(b[0][-1].item() == EOS_IDX for b in beams):
            break

    best_seq, best_logp = beams[0]
    return best_seq.tolist()


def translate_sentence(model, src_text: str, method: str = "beam", beam_size: int = 4) -> str:
    """Translate a sentence"""
    ids = encode_src(src_text, MAX_SEQ_LEN)
    src_tensor = torch.tensor(ids, dtype=torch.long).unsqueeze(0)

    if method == "beam":
        pred_ids = beam_search_decode(model, src_tensor, beam_size=beam_size)
    else:
        pred_ids = greedy_decode(model, src_tensor)

    return decode_tgt_ids(pred_ids)


# Quick qualitative test
if test_src:
    print("\n=== Quick Translation Test ===")
    print("SRC:", test_src[0])
    print("REF:", test_tgt[0])
    print("PRED (beam):", translate_sentence(model, test_src[0], method="beam", beam_size=4))





=== Quick Translation Test ===
SRC: پبلک ٹرانسپورٹ ایک بدنامی ہے۔
REF: Public transport is a disgrace.
PRED (beam): Public transport is a stigma.


In [ ]:
# ============================================================================
# CELL 18: COMPREHENSIVE EVALUATION
# ============================================================================

def comprehensive_evaluation(model, src_texts: List[str], tgt_texts: List[str],
                            max_samples: int = None):
    """Evaluate with multiple metrics"""
    model.eval()
    preds = []
    refs = []

    n_total = len(src_texts)
    n_samples = min(n_total, max_samples) if max_samples else n_total

    for i in range(n_samples):
        src = src_texts[i]
        ref = tgt_texts[i]

        pred = translate_sentence(model, src, method="beam", beam_size=4)

        preds.append(pred)
        refs.append(ref)

        if (i + 1) % 50 == 0 or (i + 1) == n_samples:
            print(f"Processed {i+1}/{n_samples} examples...")

    # Calculate metrics
    bleu = BLEU()
    chrf = CHRF()
    ter = TER()

    bleu_score = bleu.corpus_score(preds, [refs])
    chrf_score = chrf.corpus_score(preds, [refs])
    ter_score = ter.corpus_score(preds, [refs])

    return {
        'BLEU': bleu_score.score,
        'chrF': chrf_score.score,
        'TER': ter_score.score,
        'predictions': preds,
        'references': refs
    }


print("\n" + "="*80)
print("FINAL EVALUATION ON TEST SET")
print("="*80 + "\n")

results = comprehensive_evaluation(model, test_src, test_tgt, max_samples=None)

print("\n=== Final Test Metrics ===")
print(f"BLEU:  {results['BLEU']:.2f}")
print(f"chrF:  {results['chrF']:.2f}")
print(f"TER:   {results['TER']:.2f}")





FINAL EVALUATION ON TEST SET

Processed 50/9075 examples...
Processed 100/9075 examples...
Processed 150/9075 examples...
Processed 200/9075 examples...
Processed 250/9075 examples...
Processed 300/9075 examples...
Processed 350/9075 examples...
Processed 400/9075 examples...
Processed 450/9075 examples...
Processed 500/9075 examples...
Processed 550/9075 examples...
Processed 600/9075 examples...
Processed 650/9075 examples...
Processed 700/9075 examples...
Processed 750/9075 examples...
Processed 800/9075 examples...
Processed 850/9075 examples...
Processed 900/9075 examples...
Processed 950/9075 examples...
Processed 1000/9075 examples...
Processed 1050/9075 examples...
Processed 1100/9075 examples...
Processed 1150/9075 examples...
Processed 1200/9075 examples...
Processed 1250/9075 examples...
Processed 1300/9075 examples...
Processed 1350/9075 examples...
Processed 1400/9075 examples...
Processed 1450/9075 examples...
Processed 1500/9075 examples...
Processed 1550/9075 examples.

In [ ]:
# ============================================================================
# CELL 19: INSPECT EXAMPLES
# ============================================================================

def inspect_examples(model, src_texts, tgt_texts, n=15):
    """Display example translations"""
    print("\n" + "="*80)
    print("TRANSLATION EXAMPLES")
    print("="*80 + "\n")

    for i in range(min(n, len(src_texts))):
        src = src_texts[i]
        ref = tgt_texts[i]
        pred = translate_sentence(model, src, method="beam", beam_size=4)

        print(f"[Example {i+1}]")
        print(f"SRC:  {src}")
        print(f"REF:  {ref}")
        print(f"PRED: {pred}")
        print("-" * 80)

inspect_examples(model, test_src, test_tgt, n=15)





TRANSLATION EXAMPLES

[Example 1]
SRC:  پبلک ٹرانسپورٹ ایک بدنامی ہے۔
REF:  Public transport is a disgrace.
PRED: Public transport is a stigma.
--------------------------------------------------------------------------------
[Example 2]
SRC:  ‘جب آپ مالک کے گھر آجائیں اور یہ کرنا شروع کردیں تو یہ بہت خوفناک ہے۔
REF:  ‘It’s terrible when you come to the house of the Lord and start doing this’
PRED: ‘When you follow the house and start doing it, it’s so awful.
--------------------------------------------------------------------------------
[Example 3]
SRC:  میں زور دے گا کہ پوری سی ایف پی پر نظر ثانی کی جانی چاہئے ، خاص طور پر مچھلی کے ذخیروں کے تحفظ کے حصے میں۔
REF:  I would stress that the whole of the CFP must be revised, particularly the part on the protection of fish stocks.
PRED: I would stress that the whole CFP should be revised, especially in part of the protection of fish stocks.
--------------------------------------------------------------------------------
[Example 4]
SRC: 

In [ ]:
# ============================================================================
# CELL 20: SAVE FINAL MODEL AND RESULTS
# ============================================================================

# Save final model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'src_vocab_size': SRC_VOCAB_SIZE,
        'tgt_vocab_size': TGT_VOCAB_SIZE,
        'embed_dim': EMBED_DIM,
        'n_head': N_HEAD,
        'ffn_dim': FFN_DIM,
        'num_encoder_layers': NUM_ENCODER_LAYERS,
        'num_decoder_layers': NUM_DECODER_LAYERS,
        'dropout': DROPOUT,
    },
    'test_results': {
        'bleu': results['BLEU'],
        'chrf': results['chrF'],
        'ter': results['TER'],
    }
}, 'final_model.pt')

print("\n✓ Final model saved as 'final_model.pt'")

# Save translations
with open('test_translations.txt', 'w', encoding='utf-8') as f:
    for src, ref, pred in zip(test_src, test_tgt, results['predictions']):
        f.write(f"SRC: {src}\n")
        f.write(f"REF: {ref}\n")
        f.write(f"PRED: {pred}\n")
        f.write("-" * 80 + "\n")

print("✓ Test translations saved as 'test_translations.txt'")

print("\n" + "="*80)
print("ALL DONE!")
print("="*80)


✓ Final model saved as 'final_model.pt'
✓ Test translations saved as 'test_translations.txt'

ALL DONE!
